In [99]:
import pandas as pd
import numpy as np
import json
import re
from ast import literal_eval
import importlib
from dotenv import load_dotenv
load_dotenv()
import os
os.chdir(os.getenv('PARENT_DIR'))

## Preprocess

In [100]:
def add_space_around_punctuation(text):
    # Except for '-'
    # Ensure space before punctuation
    text = re.sub(r'(\S)([.,!?\(\)\"\';:+/]+)', r'\1 \2', text)
    # Ensure space after punctuation
    text = re.sub(r'([.,!?\(\)\"\';:+/]+)(\S)', r'\1 \2', text)
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    # Ensure punctuation sequences like '...' are split into spaced dots
    text = re.sub(r'([.]{2,})', lambda m: ' '.join(m.group(1)), text)
    return text.strip()

### Check mismatches after annotation

In [101]:
def get_google_sheet(sheet_id: str, sheet_gid: str) -> pd.DataFrame:
	"""
	Downloads a specific sheet from a Google Sheet into a pandas DataFrame.

	Args:
		sheet_id: The ID of the Google Sheet.
		sheet_gid: The GID of the specific sheet to download.

	Returns:
		A pandas DataFrame containing the data from the specified sheet.
	"""
	url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={sheet_gid}'
	df = pd.read_csv(url)
	return df
google_sheet_id = '1_ZEFErp75wDNkXIvYflxv1d4nAQZrMlg_Vin4cgpyyY'  # Replace with your actual Google Sheet ID

gid_hotel_eng_train = '1363651964'  # Replace with the actual GID for the English sheet
gid_hotel_eng_test = '1183495517'
gid_hotel_eng_dev = '390220786'
gid_hotel_sunda_train = '557914217'
gid_hotel_sunda_dev = '736485835'
gid_hotel_sunda_test = '1775717233'
gid_hotel_jav_train = '435666309'
gid_hotel_jav_dev = '510232916'
gid_hotel_jav_test = '166575652'
gid_hotel_min_train = '1361766185'
gid_hotel_min_dev = '72866657'
gid_hotel_min_test = '709168099'
gid_hotel_mad_train = '812663474'
gid_hotel_mad_dev = '687076406'
gid_hotel_mad_test = '1672845156'

gid_hoasa_eng_train = '495295155'
gid_hoasa_eng_dev = '1059581700'
gid_hoasa_eng_test = '275868078'
split = 'train'
lang = 'indo'
lang_target = 'min'
dataset_type = 'hotel'
dataset_folder = f'mvp_aos'
try:
	df_translated_correction = get_google_sheet(google_sheet_id, globals()[f'gid_{dataset_type}_{lang_target}_{split}'])
	print("Successfully loaded data from the specific sheet:")
except Exception as e:
	print(f"An error occurred: {e}")
	print("Please ensure the Google Sheet is shared correctly and the IDs are correct.")

Successfully loaded data from the specific sheet:


In [102]:
with open(os.path.join('dataset', 'hoasa_hotel', 'indo', 'mvp_aos', 'train.json'), 'r', encoding='utf-8') as f:
    dataset_original = json.load(f)
dataset_original[:5]

[{'sentence_id': 0,
  'instance_id': 0,
  'input': 'kamar saya ada kendala di ac tidak berfungsi optimal . dan juga wifi koneksi kurang stabil . [A] [O] [S]',
  'target': '[A] ac [O] tidak berfungsi optimal [S] negative [SSEP] [A] wifi koneksi [O] kurang stabil [S] negative',
  'element_order': 'aos',
  'task_elements': 'aos',
  'dataset_type': 'hotel_reviews'},
 {'sentence_id': 1,
  'instance_id': 1,
  'input': 'tempatnya bagus . kolam renangnya bersih . [A] [O] [S]',
  'target': '[A] tempatnya [O] bagus [S] positive [SSEP] [A] kolam renangnya [O] bersih [S] positive',
  'element_order': 'aos',
  'task_elements': 'aos',
  'dataset_type': 'hotel_reviews'},
 {'sentence_id': 2,
  'instance_id': 2,
  'input': 'oke banget , tetapi ac nya tidak bisa diatur suhu nya . [A] [O] [S]',
  'target': '[A] ac nya [O] tidak bisa diatur suhu nya [S] negative [SSEP] [A] null [O] oke banget [S] positive',
  'element_order': 'aos',
  'task_elements': 'aos',
  'dataset_type': 'hotel_reviews'},
 {'sentence

#### Check missing index in translation (after hoasa fix)

In [103]:
from typing import List, Dict
import re
def parse_absa_string(text: str) -> List[Dict[str, str]]:
	"""
	Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
	Each dictionary contains the tag as the key and the corresponding value.
	For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
	[{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
	{'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

	Args:
		text (str): ABSA string output to be parsed.

	Returns:
		List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

	"""
	pattern = r"\[(\w+)\]\s*([^[]+)"
	matches = re.findall(pattern, text)

	result = []
	current_dict = {}

	for tag, content in matches:
		if tag == "SSEP":  # Sentence separator -> Start a new dictionary
			result.append(current_dict)
			current_dict = {}
		else:
			current_dict[tag] = content.strip()

	if current_dict:  # Append the last sentence if it exists
		result.append(current_dict)

	return result


In [104]:
result1 = parse_absa_string('[A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive [SSEP] [A] fasilitas [O] nyaman [S] negative')
result2 = parse_absa_string('[A] fasilitas [O] nyaman [S] positive [SSEP] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] negative')
# Convert to set of tuples for easier comparison
set1 = set((d['A'], d['O'], d['S']) for d in result1)
set2 = set((d['A'], d['O'], d['S']) for d in result2)
set1 == set2, result1 == result2

(True, False)

In [105]:
df_translated_correction

,sentence_id,input,input_min,input_min_corrected,target_format,target_format_min,target_format_min_corrected,aspect_or_opinion_not_in_input,mismatch_notes_format,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,0,kamar saya ada kendala di ac tidak berfungsi o...,kamar ambo ado kandala di ac indak bakarajo op...,NaN,[A] ac [O] tidak berfungsi optimal [S] negativ...,[A] ac [O] indak bakarajo optimal [S] negative...,NaN,False,NaN,NaN,NaN,NaN
1,1,tempatnya bagus . kolam renangnya bersih . [A]...,tampeknyo rancak . kolam ranangnyo barasiah . ...,NaN,[A] tempatnya [O] bagus [S] positive\n[A] kola...,[A] tampeknyo [O] rancak [S] positive\n[A] kol...,NaN,False,NaN,NaN,NaN,NaN
2,2,"oke banget , tetapi ac nya tidak bisa diatur s...","oke bana , tapi ac nyo indak bisa diatur suhun...",NaN,[A] ac nya [O] tidak bisa diatur suhu nya [S] ...,[A] ac nyo [O] indak bisa diatur suhunyo [S] n...,NaN,False,NaN,NaN,NaN,NaN
3,3,keren . nyaman semuanya . [A] [O] [S],mantap . nyaman kasadonyo . [A] [O] [S],NaN,[A] semuanya [O] nyaman [S] positive\n[A] null...,[A] kasadonyo [O] nyaman [S] positive\n[A] nul...,NaN,False,NaN,NaN,NaN,NaN
4,4,"tidak dapat snack . setelah di keluhan , baru ...","indak dapek sanok . sasudah dikaluan , baru di...","indak dapek snack . sudah disabuik, baru diagi...",[A] snack [O] tidak dapat [S] negative,[A] sanok [O] indak dapek [S] negative,[A] snack [O] indak dapek [S] negative,False,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2477,2495,wifi kurang joss . [A] [O] [S],wifi nyo indak kancang . [A] [O] [S],NaN,[A] wifi [O] kurang joss [S] negative,[A] wifi nyo [O] indak kancang [S] negative,NaN,False,NaN,NaN,NaN,NaN
2478,2496,"kamar cukup bersih , hanya sempit , . [A] [O] [S]","kamar cukuik barasiah , hanyo sampik . [A] [O]...",NaN,[A] kamar [O] cukup bersih [S] positive\n[A] k...,[A] kamar [O] cukuik barasiah [S] positive\n[A...,NaN,False,NaN,NaN,NaN,NaN
2479,2497,"nyaman , bersih , dan pelayananya sangat ramah...","nyaman , barasiah , jo palayanannyo ramah bana...",NaN,[A] pelayananya [O] sangat ramah [S] positive\...,[A] palayanannyo [O] ramah bana [S] positive\n...,NaN,False,NaN,NaN,NaN,NaN
2480,2498,sangat kecewa dengan kamar dan pelayanan stafn...,sangek kecewa jo kamar jo pelayanan stafnyo ! ...,NaN,[A] kamar [O] sangat kecewa [S] negative\n[A] ...,[A] kamar [O] sangek kecewa [S] negative\n[A] ...,NaN,False,NaN,NaN,NaN,NaN


#### Continue checking mistakes in opinion and aspect mismatch

In [106]:
df_translated_correction

,sentence_id,input,input_min,input_min_corrected,target_format,target_format_min,target_format_min_corrected,aspect_or_opinion_not_in_input,mismatch_notes_format,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,0,kamar saya ada kendala di ac tidak berfungsi o...,kamar ambo ado kandala di ac indak bakarajo op...,NaN,[A] ac [O] tidak berfungsi optimal [S] negativ...,[A] ac [O] indak bakarajo optimal [S] negative...,NaN,False,NaN,NaN,NaN,NaN
1,1,tempatnya bagus . kolam renangnya bersih . [A]...,tampeknyo rancak . kolam ranangnyo barasiah . ...,NaN,[A] tempatnya [O] bagus [S] positive\n[A] kola...,[A] tampeknyo [O] rancak [S] positive\n[A] kol...,NaN,False,NaN,NaN,NaN,NaN
2,2,"oke banget , tetapi ac nya tidak bisa diatur s...","oke bana , tapi ac nyo indak bisa diatur suhun...",NaN,[A] ac nya [O] tidak bisa diatur suhu nya [S] ...,[A] ac nyo [O] indak bisa diatur suhunyo [S] n...,NaN,False,NaN,NaN,NaN,NaN
3,3,keren . nyaman semuanya . [A] [O] [S],mantap . nyaman kasadonyo . [A] [O] [S],NaN,[A] semuanya [O] nyaman [S] positive\n[A] null...,[A] kasadonyo [O] nyaman [S] positive\n[A] nul...,NaN,False,NaN,NaN,NaN,NaN
4,4,"tidak dapat snack . setelah di keluhan , baru ...","indak dapek sanok . sasudah dikaluan , baru di...","indak dapek snack . sudah disabuik, baru diagi...",[A] snack [O] tidak dapat [S] negative,[A] sanok [O] indak dapek [S] negative,[A] snack [O] indak dapek [S] negative,False,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2477,2495,wifi kurang joss . [A] [O] [S],wifi nyo indak kancang . [A] [O] [S],NaN,[A] wifi [O] kurang joss [S] negative,[A] wifi nyo [O] indak kancang [S] negative,NaN,False,NaN,NaN,NaN,NaN
2478,2496,"kamar cukup bersih , hanya sempit , . [A] [O] [S]","kamar cukuik barasiah , hanyo sampik . [A] [O]...",NaN,[A] kamar [O] cukup bersih [S] positive\n[A] k...,[A] kamar [O] cukuik barasiah [S] positive\n[A...,NaN,False,NaN,NaN,NaN,NaN
2479,2497,"nyaman , bersih , dan pelayananya sangat ramah...","nyaman , barasiah , jo palayanannyo ramah bana...",NaN,[A] pelayananya [O] sangat ramah [S] positive\...,[A] palayanannyo [O] ramah bana [S] positive\n...,NaN,False,NaN,NaN,NaN,NaN
2480,2498,sangat kecewa dengan kamar dan pelayanan stafn...,sangek kecewa jo kamar jo pelayanan stafnyo ! ...,NaN,[A] kamar [O] sangat kecewa [S] negative\n[A] ...,[A] kamar [O] sangek kecewa [S] negative\n[A] ...,NaN,False,NaN,NaN,NaN,NaN


In [107]:
df_translated_correction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2482 entries, 0 to 2481
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   sentence_id                     2482 non-null   int64  
 1   input                           2482 non-null   object 
 2   input_min                       2482 non-null   object 
 3   input_min_corrected             467 non-null    object 
 4   target_format                   2482 non-null   object 
 5   target_format_min               2480 non-null   object 
 6   target_format_min_corrected     464 non-null    object 
 7   aspect_or_opinion_not_in_input  2482 non-null   bool   
 8   mismatch_notes_format           0 non-null      float64
 9   Unnamed: 9                      0 non-null      float64
 10  Unnamed: 10                     0 non-null      float64
 11  Unnamed: 11                     111 non-null    object 
dtypes: bool(1), float64(3), int64(1), 

In [108]:
df_translated_correction[f'target_format_{lang_target}'].fillna('[A] [O] [S]', inplace=True)

C:\Users\HP\AppData\Local\Temp\ipykernel_38500\3226597203.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_translated_correction[f'target_format_{lang_target}'].fillna('[A] [O] [S]', inplace=True)


In [109]:
df_translated_correction[f'input_{lang_target}'].apply(lambda x: x.replace('’', "'") if isinstance(x, str) else print(f"Non-string value found: {x}"))

0       kamar ambo ado kandala di ac indak bakarajo op...
1       tampeknyo rancak . kolam ranangnyo barasiah . ...
2       oke bana , tapi ac nyo indak bisa diatur suhun...
3                 mantap . nyaman kasadonyo . [A] [O] [S]
4       indak dapek sanok . sasudah dikaluan , baru di...
                              ...                        
2477                 wifi nyo indak kancang . [A] [O] [S]
2478    kamar cukuik barasiah , hanyo sampik . [A] [O]...
2479    nyaman , barasiah , jo palayanannyo ramah bana...
2480    sangek kecewa jo kamar jo pelayanan stafnyo ! ...
2481    sayangnyo , aia angeknyo indak talalu angek , ...
Name: input_min, Length: 2482, dtype: object

In [110]:
df_translated_correction[f'input_{lang_target}_corrected'] = df_translated_correction[f'input_{lang_target}_corrected'].apply(lambda x: x.replace('’', "'") if isinstance(x, str) else x)
df_translated_correction[f'target_format_{lang_target}_corrected'] = df_translated_correction[f'target_format_{lang_target}_corrected'].apply(lambda x: x.replace('’', "'") if isinstance(x, str) else x)
df_translated_correction[f'input_{lang_target}'] = df_translated_correction[f'input_{lang_target}'].apply(lambda x: x.replace('’', "'"))
df_translated_correction[f'target_format_{lang_target}'] = df_translated_correction[f'target_format_{lang_target}'].apply(lambda x: x.replace('’', "'"))

In [111]:
list(df_translated_correction.loc[df_translated_correction['sentence_id'] == 433, f'input_{lang_target}_corrected'])

[nan]

In [112]:
# Strip strings of all _corrected columns
df_translated_correction[f'input_{lang_target}_corrected'] = df_translated_correction[f'input_{lang_target}_corrected'].apply(lambda x: x.strip() if isinstance(x, str) else x)
df_translated_correction[f'target_format_{lang_target}_corrected'] = df_translated_correction[f'target_format_{lang_target}_corrected'].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Set to pandas nan if empty string
df_translated_correction[f'input_{lang_target}_corrected'] = df_translated_correction[f'input_{lang_target}_corrected'].apply(lambda x: x if x != '' else np.nan)
df_translated_correction[f'target_format_{lang_target}_corrected'] = df_translated_correction[f'target_format_{lang_target}_corrected'].apply(lambda x: x if x != '' else np.nan)

In [113]:
# Fill _corrected columns with input_eng if null
df_translated_correction[f'input_{lang_target}_corrected'] = df_translated_correction[f'input_{lang_target}_corrected'].fillna(df_translated_correction[f'input_{lang_target}'])
df_translated_correction[f'target_format_{lang_target}_corrected'] = df_translated_correction[f'target_format_{lang_target}_corrected'].fillna(df_translated_correction[f'target_format_{lang_target}'])

In [114]:
df_translated_correction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2482 entries, 0 to 2481
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   sentence_id                     2482 non-null   int64  
 1   input                           2482 non-null   object 
 2   input_min                       2482 non-null   object 
 3   input_min_corrected             2482 non-null   object 
 4   target_format                   2482 non-null   object 
 5   target_format_min               2482 non-null   object 
 6   target_format_min_corrected     2482 non-null   object 
 7   aspect_or_opinion_not_in_input  2482 non-null   bool   
 8   mismatch_notes_format           0 non-null      float64
 9   Unnamed: 9                      0 non-null      float64
 10  Unnamed: 10                     0 non-null      float64
 11  Unnamed: 11                     111 non-null    object 
dtypes: bool(1), float64(3), int64(1), 

In [115]:
def check_mismatches_triplet_format_sent_id(outputs_text):
	mismatch_indexes = []
	mismatch_notes = {}
	for key, instance in outputs_text.items():
		translated_text = add_space_around_punctuation(instance['translated_text'].lower()).strip()
		mismatched = False
		for triplet in instance['translated_triplets']:
			aspect_term = add_space_around_punctuation(triplet['aspect_term'].lower()).strip()
			opinion_term = add_space_around_punctuation(triplet['opinion_term'].lower()).strip()
			if aspect_term not in translated_text and aspect_term != 'null':
				print(f"Mismatch in instance sentence_id {instance['sentence_id']} key {key}: aspect_term '{aspect_term}' not found in {translated_text}")
				mismatch_notes[key] = mismatch_notes.get(key, []) + [f"aspect_term '{aspect_term}' not found"]
				mismatched = True
			if opinion_term not in translated_text and opinion_term != 'null':
				print(f"Mismatch in instance sentence_id {instance['sentence_id']} key {key}: opinion_term '{opinion_term}' not found in {translated_text}")
				mismatch_notes[key] = mismatch_notes.get(key, []) + [f"opinion_term '{opinion_term}' not found"]
				mismatched = True
		if mismatched:
			mismatch_indexes.append(key)
	print(f"Total mismatches found: {len(mismatch_indexes)}")
	return mismatch_indexes, mismatch_notes

In [116]:
outputs_text_correction = {}
for idx, row in df_translated_correction.iterrows():
	outputs_text_correction[idx] = {
		'sentence_id': row['sentence_id'],
		'translated_text': row[f'input_{lang_target}_corrected'],
		'translated_triplets': parse_absa_string(row[f'target_format_{lang_target}_corrected'].replace('\n', ' [SSEP] '))
	}
	# Change the keys of translated_triplets from A, O, S to aspect_term, opinion_term, sentiment_polarity
	for triplet in outputs_text_correction[idx]['translated_triplets']:
		triplet['aspect_term'] = triplet.pop('A', '')
		triplet['opinion_term'] = triplet.pop('O', '')
		triplet['sentiment_polarity'] = triplet.pop('S', '')

In [117]:
len(outputs_text_correction)

2482

In [118]:
outputs_text_correction[372]

{'sentence_id': 373,
 'translated_text': 'sasuai jo nan dibayia . harago total untuak 3 kamar 1 malam cuma 324000 ( pakai promo airy ) jo di daerah kemang ! lokasi rancak , dakek kemang . kamar lumayan gadang tapi agak indak sasuai jo foto . dek bangunannyo rumah , ukuran kamar babeda-beda . kolam ranang rancak . indak ado aia angek untuak mandi jo wifi nyo indak jalan ( padahal username jo password lah batua ) . [A] [O] [S]',
 'translated_triplets': [{'aspect_term': 'harago',
   'opinion_term': 'total untuak 3 kamar 1 malam cuma 324000 ( pakai promo airy ) jo di daerah kemang',
   'sentiment_polarity': 'positive'},
  {'aspect_term': 'lokasi',
   'opinion_term': 'rancak',
   'sentiment_polarity': 'positive'},
  {'aspect_term': 'lokasi',
   'opinion_term': 'dakek kemang',
   'sentiment_polarity': 'positive'},
  {'aspect_term': 'kamar',
   'opinion_term': 'lumayan gadang',
   'sentiment_polarity': 'positive'},
  {'aspect_term': 'kamar',
   'opinion_term': 'indak sasuai jo foto . dek bang

In [119]:
mismatch_indexes, mismatch_notes = check_mismatches_triplet_format_sent_id(outputs_text_correction)

Total mismatches found: 0


In [120]:
df_translated_correction['aspect_or_opinion_not_in_input'] = [idx in mismatch_indexes for idx in df_translated_correction.index]
df_translated_correction['mismatch_notes'] = [mismatch_notes.get(idx, []) for idx in df_translated_correction.index]
df_translated_correction['mismatch_notes_format'] = df_translated_correction['mismatch_notes'].apply(lambda x: '\n'.join(x))

In [121]:
os.makedirs(f'temp/translation_output/{lang_target}/{dataset_folder}', exist_ok=True)
df_translated_correction[['aspect_or_opinion_not_in_input', 'mismatch_notes_format']].to_csv(f'temp/translation_output/{lang_target}/{dataset_folder}/translated_dataset_{lang}_{split}_correction_mismatches.csv', index=False)

In [122]:
df_translated_correction.shape

(2482, 13)

### Store to dataset

#### Check missing index in translation (after hoasa fix)

In [123]:
from typing import List, Dict
import re
def parse_absa_string(text: str) -> List[Dict[str, str]]:
	"""
	Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
	Each dictionary contains the tag as the key and the corresponding value.
	For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
	[{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
	{'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

	Args:
		text (str): ABSA string output to be parsed.

	Returns:
		List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

	"""
	pattern = r"\[(\w+)\]\s*([^[]+)"
	matches = re.findall(pattern, text)

	result = []
	current_dict = {}

	for tag, content in matches:
		if tag == "SSEP":  # Sentence separator -> Start a new dictionary
			result.append(current_dict)
			current_dict = {}
		else:
			current_dict[tag] = content.strip()

	if current_dict:  # Append the last sentence if it exists
		result.append(current_dict)

	return result


In [124]:
def check_mismatches_triplet_format_sent_id(outputs_text):
	mismatch_indexes = []
	mismatch_notes = {}
	for key, instance in outputs_text.items():
		translated_text = add_space_around_punctuation(instance['translated_text'].lower()).strip()
		mismatched = False
		for triplet in instance['translated_triplets']:
			aspect_term = add_space_around_punctuation(triplet['aspect_term'].lower()).strip()
			opinion_term = add_space_around_punctuation(triplet['opinion_term'].lower()).strip()
			if aspect_term not in translated_text and aspect_term != 'null':
				print(f"Mismatch in instance sentence_id {instance['sentence_id']} key {key}: aspect_term '{aspect_term}' not found in {translated_text}")
				mismatch_notes[key] = mismatch_notes.get(key, []) + [f"aspect_term '{aspect_term}' not found"]
				mismatched = True
			if opinion_term not in translated_text and opinion_term != 'null':
				print(f"Mismatch in instance sentence_id {instance['sentence_id']} key {key}: opinion_term '{opinion_term}' not found in {translated_text}")
				mismatch_notes[key] = mismatch_notes.get(key, []) + [f"opinion_term '{opinion_term}' not found"]
				mismatched = True
		if mismatched:
			mismatch_indexes.append(key)
	print(f"Total mismatches found: {len(mismatch_indexes)}")
	return mismatch_indexes, mismatch_notes

#### Continue checking mistakes in opinion and aspect mismatch

In [134]:
def get_google_sheet(sheet_id: str, sheet_gid: str) -> pd.DataFrame:
	"""
	Downloads a specific sheet from a Google Sheet into a pandas DataFrame.

	Args:
		sheet_id: The ID of the Google Sheet.
		sheet_gid: The GID of the specific sheet to download.

	Returns:
		A pandas DataFrame containing the data from the specified sheet.
	"""
	url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={sheet_gid}'
	df = pd.read_csv(url)
	return df
google_sheet_id = '1_ZEFErp75wDNkXIvYflxv1d4nAQZrMlg_Vin4cgpyyY'  # Replace with your actual Google Sheet ID

gid_hotel_eng_train = '1363651964'  # Replace with the actual GID for the English sheet
gid_hotel_eng_test = '1183495517'
gid_hotel_eng_dev = '390220786'
gid_hotel_sunda_train = '557914217'
gid_hotel_sunda_dev = '736485835'
gid_hotel_sunda_test = '1775717233'
gid_hotel_jav_train = '435666309'
gid_hotel_jav_dev = '510232916'
gid_hotel_jav_test = '166575652'
gid_hotel_min_train = '1361766185'
gid_hotel_min_dev = '72866657'
gid_hotel_min_test = '709168099'
gid_hotel_mad_train = '812663474'
gid_hotel_mad_dev = '687076406'
gid_hotel_mad_test = '1672845156'

gid_hoasa_eng_train = '495295155'
gid_hoasa_eng_dev = '1059581700'
gid_hoasa_eng_test = '275868078'
splits = ['train', 'dev', 'test']
split = 'test'
lang = 'indo'
lang_target = 'min'
dataset_type = 'hotel'
dataset_folder = f'mvp_aos'
try:
	df_translated_correction = get_google_sheet(google_sheet_id, globals()[f'gid_{dataset_type}_{lang_target}_{split}'])
	print("Successfully loaded data from the specific sheet:")
except Exception as e:
	print(f"An error occurred: {e}")
	print("Please ensure the Google Sheet is shared correctly and the IDs are correct.")

df_translated_correction[f'target_format_{lang_target}'].fillna('[A] [O] [S]', inplace=True)
df_translated_correction[f'input_{lang_target}_corrected'] = df_translated_correction[f'input_{lang_target}_corrected'].apply(lambda x: x.replace('’', "'") if isinstance(x, str) else x)
df_translated_correction[f'target_format_{lang_target}_corrected'] = df_translated_correction[f'target_format_{lang_target}_corrected'].apply(lambda x: x.replace('’', "'") if isinstance(x, str) else x)
df_translated_correction[f'input_{lang_target}'] = df_translated_correction[f'input_{lang_target}'].apply(lambda x: x.replace('’', "'"))
df_translated_correction[f'target_format_{lang_target}'] = df_translated_correction[f'target_format_{lang_target}'].apply(lambda x: x.replace('’', "'"))

# Strip strings of all _corrected columns
df_translated_correction[f'input_{lang_target}_corrected'] = df_translated_correction[f'input_{lang_target}_corrected'].apply(lambda x: x.strip() if isinstance(x, str) else x)
df_translated_correction[f'target_format_{lang_target}_corrected'] = df_translated_correction[f'target_format_{lang_target}_corrected'].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Set to pandas nan if empty string
df_translated_correction[f'input_{lang_target}_corrected'] = df_translated_correction[f'input_{lang_target}_corrected'].apply(lambda x: x if x != '' else np.nan)
df_translated_correction[f'target_format_{lang_target}_corrected'] = df_translated_correction[f'target_format_{lang_target}_corrected'].apply(lambda x: x if x != '' else np.nan)

# Fill _corrected columns with input_eng if null
df_translated_correction[f'input_{lang_target}_corrected'] = df_translated_correction[f'input_{lang_target}_corrected'].fillna(df_translated_correction[f'input_{lang_target}'])
df_translated_correction[f'target_format_{lang_target}_corrected'] = df_translated_correction[f'target_format_{lang_target}_corrected'].fillna(df_translated_correction[f'target_format_{lang_target}'])

outputs_text_correction = {}
for idx, row in df_translated_correction.iterrows():
	outputs_text_correction[idx] = {
		'sentence_id': row['sentence_id'],
		'translated_text': row[f'input_{lang_target}_corrected'],
		'translated_triplets': parse_absa_string(row[f'target_format_{lang_target}_corrected'].replace('\n', ' [SSEP] '))
	}
	# Change the keys of translated_triplets from A, O, S to aspect_term, opinion_term, sentiment_polarity
	for triplet in outputs_text_correction[idx]['translated_triplets']:
		triplet['aspect_term'] = triplet.pop('A', '')
		triplet['opinion_term'] = triplet.pop('O', '')
		triplet['sentiment_polarity'] = triplet.pop('S', '')

mismatch_indexes, mismatch_notes = check_mismatches_triplet_format_sent_id(outputs_text_correction)

df_translated_correction['aspect_or_opinion_not_in_input'] = [idx in mismatch_indexes for idx in df_translated_correction.index]
df_translated_correction['mismatch_notes'] = [mismatch_notes.get(idx, []) for idx in df_translated_correction.index]
df_translated_correction['mismatch_notes_format'] = df_translated_correction['mismatch_notes'].apply(lambda x: '\n'.join(x))

new_dataset = [] # New dataset for mvp_aos format
for idx, row in df_translated_correction.iterrows():
	sentence_id = row['sentence_id']
	instance_id = row['sentence_id'] * 5
	input = row[f'input_{lang_target}_corrected']
	target_format = row[f'target_format_{lang_target}_corrected']
	target_format = target_format.strip().replace('\n', ' [SSEP] ')
	element_order = 'aos'
	task_elements = 'aos'
	dataset_type = 'hotel_reviews' if dataset_type == 'hotel' else dataset_type
	new_dataset.append({
		'sentence_id': sentence_id,
		'instance_id': instance_id,
		'input': input,
		'target': target_format,
		'element_order': element_order,
		'task_elements': task_elements,
		'dataset_type': dataset_type
	})

save_path = f'dataset\\{'hotel_reviews' if dataset_type == 'hotel' else dataset_type}\\{lang_target}\\{dataset_folder}\\{split}.json'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, 'w', encoding='utf-8') as f:
	json.dump(new_dataset, f, ensure_ascii=False, indent=4)
print(f"Saved new dataset to {save_path} with {len(new_dataset)} instances")

Successfully loaded data from the specific sheet:
Total mismatches found: 0
Saved new dataset to dataset\hotel_reviews\min\mvp_aos\test.json with 1000 instances


C:\Users\HP\AppData\Local\Temp\ipykernel_38500\4049115895.py:49: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_translated_correction[f'target_format_{lang_target}'].fillna('[A] [O] [S]', inplace=True)


Saved new dataset to dataset\hotel_reviews\min\mvp_aos\train.json with 2482 instances
